In [0]:
dbutils.fs.ls("/Volumes/workspace/lab_tuan1/raw/")


Bước 1 — Tạo không gian tên ba tầng

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dev_qai_tuan1;


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze_tuan1;
CREATE SCHEMA IF NOT EXISTS workspace.silver_tuan1;
CREATE SCHEMA IF NOT EXISTS workspace.gold_tuan1;


In [0]:
%sql
SHOW SCHEMAS IN workspace LIKE '*_tuan1';


Bước 2 — Bảng managed

2.1 Tạo bảng ở tầng gold

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold_tuan1.dim_shop_tuan1 (
  shop_id     STRING,
  shop_code   STRING,
  shop_name   STRING,
  region      STRING,
  status      STRING
) USING DELTA;

INSERT INTO workspace.gold_tuan1.dim_shop_tuan1
SELECT shop_id, shop_code, shop_name, region, status
FROM read_files('/Volumes/workspace/lab_tuan1/raw/shops.csv',
                format => 'csv', header => true);


In [0]:
%sql
SELECT COUNT(*) AS so_shop FROM workspace.gold_tuan1.dim_shop_tuan1;
SELECT * FROM workspace.gold_tuan1.dim_shop_tuan1 WHERE status = 'ACTIVE';


2.3 Ghi lại loại bảng và đường dẫn 

In [0]:
%sql
DESCRIBE EXTENDED workspace.gold_tuan1.dim_shop_tuan1;


In [0]:
%sql
DESCRIBE DETAIL workspace.gold_tuan1.dim_shop_tuan1;


Bước 3 — Bảng external

3.1 Thử tạo external table trỏ vào Volume

In [0]:
path_ext = "/Volumes/workspace/lab_tuan1/raw/ext_demo"
(spark.read.csv("/Volumes/workspace/lab_tuan1/raw/shops.csv", header=True, inferSchema=True)
     .write.mode("overwrite").format("delta").save(path_ext))
print("da ghi file vao:", path_ext)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.bronze_tuan1.raw_shops_tuan1
USING DELTA
LOCATION '/Volumes/workspace/lab_taun1/raw/ext_demo/';
